# 03 -- Strategy/regime analysis

Paired script: `analysis/regime_validation.py` -- a Python port of `MarketRegimeEngine.mqh`'s
(TASK-016) classification formula, closing that task's deferred "regime fixtures/confusion
matrix" item. Demonstrates all 7 directly-computed regime states against hand-constructed
synthetic fixtures (the same ones hand-verified in `tests/test_regime_validation.py`), plus
a small confusion-matrix example.

**No confusion matrix against independently-labelled real evidence yet** -- see final cell.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.regime_validation import Regime, build_confusion_matrix, classify

TREND_CLOSES = [104.0, 103.0, 102.0, 101.0, 100.0]
COMMON = dict(efficiency_window=4, trend_threshold=0.6, expansion_threshold=0.75,
              compression_threshold=0.25, min_efficiency=0.3, trend_slope_atr_divisor=0.5)

In [ ]:
trending_up = classify(TREND_CLOSES, [0.0, 2.0, 2.0], 1.0, ema_now=105.0, ema_prior=100.0,
                        adx_now=50.0, **COMMON, swing_agreement=1.0, direction_agree=True)
expansion_up = classify(TREND_CLOSES, [0.0, 1.0, 1.0], 2.0, ema_now=105.0, ema_prior=100.0,
                         adx_now=50.0, **COMMON, swing_agreement=1.0, direction_agree=True)
compression = classify(TREND_CLOSES, [0.0, 5.0, 5.0], 1.0, ema_now=100.0, ema_prior=100.0,
                        adx_now=50.0, **COMMON, swing_agreement=0.0, direction_agree=False)

for label, r in [("TRENDING_UP", trending_up), ("VOLATILITY_EXPANSION_UP", expansion_up),
                 ("COMPRESSION", compression)]:
    print(f"{label:28s} -> regime={r.regime.value:28s} confidence={r.confidence:.4f} (valid={r.valid})")

assert trending_up.regime == Regime.TRENDING_UP
assert expansion_up.regime == Regime.VOLATILITY_EXPANSION_UP
assert compression.regime == Regime.COMPRESSION

In [ ]:
# Small illustrative confusion matrix (synthetic labels, not real evidence).
predicted = ["TRENDING_UP", "TRENDING_UP", "RANGING", "RANGING", "TRENDING_UP"]
actual =    ["TRENDING_UP", "RANGING",     "RANGING", "RANGING", "TRENDING_UP"]
matrix = build_confusion_matrix(predicted, actual)
print(matrix)

## Confusion matrix against real, independently-labelled evidence: PENDING

No such dataset exists yet in this project. `build_confusion_matrix` above is ready to use
once one does. Also note (see `regime_validation.py`'s own docstring):
`swing_agreement`/`direction_agree` were supplied directly here, not computed from real
`MarketStructure.mqh`-equivalent logic -- that port is separate, not-yet-attempted work.